# Algorithm Benchmark

This notebook compares RSA, ML-KEM, ML-DSA, SLH-DSA, and HQC for the payment transaction security use case.

Implementation note: this notebook does not duplicate benchmark logic. It imports and visualizes functions from `benchmarks.algorithm_benchmark`, so the package module remains the single source of truth.

Notes:
- ML-KEM and HQC are KEM/encryption primitives for protecting payment data keys.
- ML-DSA and SLH-DSA are signature primitives for transaction authorization.
- If `liboqs` or `cryptography` is unavailable, the benchmark module clearly reports `simulation fallback`.

In [4]:
from pathlib import Path
import sys

project_root = Path.cwd()
if (project_root / "src").exists():
    sys.path.insert(0, str(project_root / "src"))
else:
    sys.path.insert(0, str(project_root))

from algorithms.crypto_algorithms import list_algorithm_profiles
from benchmarks.algorithm_benchmark import benchmark_results_as_dicts, CRYPTOGRAPHY_AVAILABLE, OQS_AVAILABLE

print(f"cryptography available: {CRYPTOGRAPHY_AVAILABLE}")
print(f"liboqs available: {OQS_AVAILABLE}")

cryptography available: True
liboqs available: False


## Security Scorecard

The base score is the starting cryptographic posture score before transaction-specific penalties are applied.

In [5]:
try:
    import pandas as pd
    profiles_df = pd.DataFrame(list_algorithm_profiles())
    display(profiles_df[[
        "name",
        "primary_use",
        "nist_status",
        "quantum_safe",
        "security_level",
        "base_score",
        "production_role",
    ]])
except ImportError:
    for profile in list_algorithm_profiles():
        print(profile)

,name,primary_use,nist_status,quantum_safe,security_level,base_score,production_role
0,RSA-2048,Encryption/signature,Classical legacy,False,Classical ~112-bit; vulnerable to Shor's algor...,46,Legacy baseline for comparison
1,ML-KEM-768,Key encapsulation/encryption,FIPS 203,True,NIST category 3,93,Primary PQC encryption/KEM choice
2,ML-DSA-65,Digital signature,FIPS 204,True,NIST category 3,91,Primary PQC signature choice
3,SLH-DSA-128s,Digital signature,FIPS 205,True,NIST category 1,84,Conservative backup signature option
4,HQC-128,Key encapsulation/encryption,Selected by NIST in 2025 for future standardiz...,True,NIST category 1 target,82,Backup KEM for crypto-agility


## Run Local Benchmark

Increase `iterations` for a more stable comparison. Keep it small when running on an older laptop or without native wheels installed.

In [6]:
iterations = 5
benchmark_rows = benchmark_results_as_dicts(iterations=iterations)

try:
    benchmark_df = pd.DataFrame(benchmark_rows)
    display(benchmark_df)
except NameError:
    for row in benchmark_rows:
        print(row)

,algorithm,primitive,implementation,avg_ms,p95_ms,security_score,quantum_safe,notes
0,RSA-2048,Encryption + signature,cryptography real RSA,77.884,98.510,46,False,"Legacy baseline: real RSA-OAEP key wrap, AES-G..."
1,ML-KEM-768,KEM + AEAD encryption,simulation fallback,0.243,1.175,93,True,Primary PQC encryption/KEM path for protecting...
2,ML-DSA-65,Digital signature,simulation fallback,0.094,0.102,91,True,Primary PQC transaction authorization signature.
3,SLH-DSA-128s,Digital signature,simulation fallback,0.172,0.174,84,True,Hash-based backup signature; useful for crypto...
4,HQC-128,KEM + AEAD encryption,simulation fallback,0.012,0.022,82,True,Code-based backup KEM candidate with different...


## Payment Security Interpretation

Use **ML-KEM-768** to protect payment payload data keys and **ML-DSA-65** to sign transaction authorization messages. Keep **SLH-DSA** and **HQC** as backup families for crypto-agility.